<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/C5_results_lattice4_kron_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# Colab — C.5 Diagnostic anti « branched polymer » (v3, code complet)
# 2D, patch borné B_R (R = c t), cellules l_p*l_p, connectivité 4 contacts (N,S,E,O)
# + stochastique UV (percolation des liens) + overlaps (conductances entières)
# + résidu DSI A_eff(Δτ) "pic → chute → résidu"
# + coarse-graining "Kigami-like" via Kron reduction (Schur complement) régularisée
# + C.5 calculé sur la LCC (robuste en fragmentation UV)
# + aide automatique au choix de R (fonction recommend_R_list)
#
# Sorties :
#   - C5_results_lattice4_kron_v3.csv
#   - C5_summary_lattice4_kron_v3.csv
# ==========================================

# ---- deps ----
try:
    import scipy
    from scipy.sparse import coo_matrix, identity
    from scipy.sparse.linalg import splu
except Exception:
    !pip -q install scipy
    import scipy
    from scipy.sparse import coo_matrix, identity
    from scipy.sparse.linalg import splu

import math
from collections import deque, defaultdict
import numpy as np
import pandas as pd


# ============================================================
# 0) Outils graph / composantes / métriques
# ============================================================

def induced_subgraph(V, E, V_keep):
    Vp = set(V_keep)
    Ep = [(u, v) for (u, v) in E if (u in Vp and v in Vp and u != v)]
    return list(Vp), Ep

def adjacency_on_subset(V_subset, E_subset):
    Vset = set(V_subset)
    adj = {v: set() for v in Vset}
    for (u, v) in E_subset:
        if u in Vset and v in Vset and u != v:
            adj[u].add(v)
            adj[v].add(u)
    return adj

def connected_components(adj, nodes_subset=None):
    nodes = set(adj.keys()) if nodes_subset is None else set(nodes_subset)
    seen, comps = set(), []
    for s in list(nodes):
        if s in seen:
            continue
        q = deque([s])
        seen.add(s)
        comp = {s}
        while q:
            u = q.popleft()
            for v in adj[u]:
                if v in nodes and v not in seen:
                    seen.add(v)
                    comp.add(v)
                    q.append(v)
        comps.append(comp)
    return comps

def lcc_set(V_subset, E_subset):
    adj = adjacency_on_subset(V_subset, E_subset)
    comps = connected_components(adj, set(adj.keys()))
    if not comps:
        return set()
    return max(comps, key=len)

def metrics_full(V_subset, E_subset):
    """
    Métriques sur tout l'intérieur (utile pour diagnostiquer fragmentation) :
      b1 = |E|-|V|+c
      rho_loop = b1/|V|
      lcc_ratio
      deg_mean, deg_max
    """
    Vset = set(V_subset)
    if len(Vset) == 0:
        return dict(n=0, b1=0, rho_loop=float("nan"), lcc=0, lcc_ratio=float("nan"),
                    deg_mean=float("nan"), deg_max=float("nan"))

    adj = adjacency_on_subset(Vset, E_subset)
    comps = connected_components(adj, Vset)
    c = len(comps)
    n = len(Vset)
    m = sum(1 for (u, v) in E_subset if u in Vset and v in Vset and u != v)
    b1 = m - n + c
    rho = b1 / n

    lcc = max((len(comp) for comp in comps), default=0)
    lcc_ratio = lcc / n

    degs = [len(adj[v]) for v in Vset]
    deg_mean = float(np.mean(degs)) if degs else float("nan")
    deg_max  = float(np.max(degs))  if degs else float("nan")

    return dict(n=int(n), b1=int(b1), rho_loop=float(rho),
                lcc=int(lcc), lcc_ratio=float(lcc_ratio),
                deg_mean=float(deg_mean), deg_max=float(deg_max))

def metrics_on_lcc(V_subset, E_subset):
    """
    C.5 sur la plus grande composante connexe (LCC) uniquement :
    Sur un graphe connexe, b1 = |E|-|V|+1 ; rho = b1/|V|.
    """
    L = lcc_set(V_subset, E_subset)
    n = len(L)
    if n == 0:
        return dict(n_lcc=0, b1_lcc=0, rho_loop_lcc=float("nan"))
    m = sum(1 for (u, v) in E_subset if u in L and v in L and u != v)
    b1 = m - n + 1
    rho = b1 / n
    return dict(n_lcc=int(n), b1_lcc=int(b1), rho_loop_lcc=float(rho))

def interior_nodes_by_radius(node_radii, R, margin):
    cutoff = R - margin
    return [v for v, r in node_radii.items() if r <= cutoff]

def incidence_from_conductances(V_subset, E_subset, C):
    """
    Proxy incidence/overlap : m(v)=Σ C_vw.
    """
    Vset = set(V_subset)
    if not Vset:
        return dict(m_mean=float("nan"), m_max=float("nan"))
    m = {v: 0.0 for v in Vset}
    for (u, v) in E_subset:
        if u in Vset and v in Vset:
            key = (u, v) if u < v else (v, u)
            w = float(C.get(key, 1.0))
            m[u] += w
            m[v] += w
    vals = list(m.values())
    return dict(m_mean=float(np.mean(vals)) if vals else float("nan"),
                m_max=float(np.max(vals)) if vals else float("nan"))


# ============================================================
# 1) Tessellation : lattice 2D 4-connexe dans B_R
# ============================================================

def generate_lattice4_base(ell, R, lp=1.0):
    """
    Niveau ℓ : pas s_ℓ = 2^ℓ l_p.
    Sommets : points (i*sℓ, j*sℓ) tels que x^2+y^2 ≤ R^2.
    Arêtes : 4-connexité (N,S,E,O).
    """
    s = (2**ell) * lp
    M = int(math.floor(R / s))

    coords = []
    for i in range(-M, M+1):
        x = i*s
        for j in range(-M, M+1):
            y = j*s
            if x*x + y*y <= R*R + 1e-12:
                coords.append((i, j))

    id_of = {ij: k for k, ij in enumerate(coords)}
    V = list(range(len(coords)))
    pos = {id_of[(i, j)]: (i*s, j*s) for (i, j) in coords}
    radii = {vid: math.hypot(*pos[vid]) for vid in V}
    coord_of = {id_of[(i, j)]: (i, j) for (i, j) in coords}

    E_geo = []
    for (i, j) in coords:
        u = id_of[(i, j)]
        for di, dj in [(1,0), (-1,0), (0,1), (0,-1)]:
            nb = (i+di, j+dj)
            if nb in id_of:
                v = id_of[nb]
                if u < v:
                    E_geo.append((u, v))

    return V, E_geo, pos, radii, coord_of, id_of, s

def blocks_2x2(coord_of):
    """
    Π_{ℓ→ℓ+1} géométrique : regroupement 2×2 (i,j)->(I,J)=(floor(i/2), floor(j/2)).
    """
    blocks = defaultdict(list)
    for u, (i, j) in coord_of.items():
        I, J = (i // 2), (j // 2)
        blocks[(I, J)].append(u)

    IJ_list = list(blocks.keys())
    coarse_id_of = {IJ: k for k, IJ in enumerate(IJ_list)}
    IJ_of_coarse = {k: IJ for IJ, k in coarse_id_of.items()}

    Pi = {}
    for IJ, fine_list in blocks.items():
        cid = coarse_id_of[IJ]
        for u in fine_list:
            Pi[u] = cid

    return blocks, coarse_id_of, IJ_of_coarse, Pi

def choose_anchor_per_block(blocks, coarse_id_of):
    """
    Un anchor fin par bloc (min id), utilisé comme DOF conservé pour Kron.
    Retour:
      keep_fine : list of fine ids
      coarse_of_anchor_fine : dict fine_id -> coarse_id
    """
    coarse_of_anchor_fine = {}
    for IJ, fine_list in blocks.items():
        cid = coarse_id_of[IJ]
        a = min(fine_list)
        coarse_of_anchor_fine[a] = cid
    keep_fine = list(coarse_of_anchor_fine.keys())
    return keep_fine, coarse_of_anchor_fine

def coarse_local_edges(IJ_of_coarse):
    """
    Arêtes locales coarse (4-voisins entre blocs existants).
    """
    id_of_IJ = {IJ: cid for cid, IJ in IJ_of_coarse.items()}
    E_loc = []
    for cid, (I, J) in IJ_of_coarse.items():
        for dI, dJ in [(1,0), (-1,0), (0,1), (0,-1)]:
            nb = (I+dI, J+dJ)
            if nb in id_of_IJ:
                cid2 = id_of_IJ[nb]
                if cid < cid2:
                    E_loc.append((cid, cid2))
    return E_loc

def coarse_radii_from_blocks(blocks, coarse_id_of, pos_fine):
    """
    Rayon coarse = norme du barycentre du bloc.
    """
    rad_c = {}
    for IJ, fine_list in blocks.items():
        cid = coarse_id_of[IJ]
        xs = [pos_fine[u][0] for u in fine_list]
        ys = [pos_fine[u][1] for u in fine_list]
        x = float(np.mean(xs))
        y = float(np.mean(ys))
        rad_c[cid] = math.hypot(x, y)
    return rad_c


# ============================================================
# 2) DSI A_eff(Δτ) + calibration p_keep(τ_c)=p_c + overlaps
# ============================================================

def Aeff_DSI(delta_tau, tau_c=0.08, A_peak=1.0, A_res=0.15,
             gamma_down=40.0, gamma_up=4.0, gamma_minus=6.0):
    """
    A_eff(Δτ):
    - pic à τ_c (A_peak)
    - IR: → 0
    - UV: chute puis plateau résiduel A_res
    """
    x = max(float(delta_tau), 1e-12)
    if x >= tau_c:
        return A_peak * math.exp(-gamma_minus * (x - tau_c))
    d = (tau_c - x)
    return A_peak * math.exp(-gamma_down * d) + A_res * (1.0 - math.exp(-gamma_up * d))

def p_keep_from_delta_tau(delta_tau, tau_c=0.08, p_c=0.5, kappa_keep=1.3, p_floor=0.0, p_ceil=1.0):
    """
    p_keep(Δτ) monotone en ln(Δτ/τ_c), calibré par p_keep(τ_c)=p_c.
    Forme : p_keep = p_c + (p_ceil-p_c)*tanh(kappa*ln(Δτ/τ_c))
    """
    x = max(float(delta_tau), 1e-12)
    z = kappa_keep * math.log(x / tau_c)
    val = p_c + (p_ceil - p_c) * math.tanh(z)
    return max(p_floor, min(p_ceil, val))

def lambda_overlap(delta_tau, tau_c=0.08,
                   lam_bg_scale=1.2, alpha_bg=0.8,
                   lam_dsi_scale=3.0, lam_max=10.0):
    """
    Intensité overlaps: fond UV + contribution DSI (via A_eff).
    """
    x = max(float(delta_tau), 1e-12)
    lam_bg = lam_bg_scale * (x ** (-alpha_bg))
    lam_dsi = lam_dsi_scale * Aeff_DSI(x, tau_c=tau_c)
    return min(lam_bg + lam_dsi, lam_max)

def apply_stochastic_edges(E_geo, rng, delta_tau, tau_c=0.08, p_c=0.5, kappa_keep=1.3):
    """
    - conserve chaque arête géométrique avec proba p_keep(Δτ)
    - C_uv = 1 + Poisson(λ_overlap(Δτ))
    """
    p_keep = p_keep_from_delta_tau(delta_tau, tau_c=tau_c, p_c=p_c, kappa_keep=kappa_keep)
    lam = lambda_overlap(delta_tau, tau_c=tau_c)
    Aeff = Aeff_DSI(delta_tau, tau_c=tau_c)

    E = []
    C = {}
    for (u, v) in E_geo:
        if rng.random() <= p_keep:
            if u > v:
                u, v = v, u
            mult = 1 + int(rng.poisson(lam))
            E.append((u, v))
            C[(u, v)] = float(mult)

    st = {"p_keep": float(p_keep), "lambda_ov": float(lam), "Aeff": float(Aeff)}
    return E, C, st


# ============================================================
# 3) Laplacien + Kron reduction régularisée
# ============================================================

def laplacian_from_conductances(n_nodes, edges, C):
    """
    Laplacien sparse CSC :
      L_uu += w, L_vv += w, L_uv -= w, L_vu -= w
    """
    rows, cols, data = [], [], []
    diag = np.zeros(n_nodes, dtype=float)

    for (u, v) in edges:
        key = (u, v) if u < v else (v, u)
        w = float(C.get(key, 1.0))
        diag[u] += w
        diag[v] += w
        rows.extend([u, v])
        cols.extend([v, u])
        data.extend([-w, -w])

    rows.extend(list(range(n_nodes)))
    cols.extend(list(range(n_nodes)))
    data.extend(diag.tolist())

    L = coo_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes)).tocsc()
    L.sum_duplicates()
    return L

def kron_reduction(L, keep_idx, leak_reg=1e-9):
    """
    Schur complement:
      L_eff = L_kk - L_ki * inv(L_ii) * L_ik
    Régularisation: L_ii + eps I pour éviter singularité (graphe percolé/déconnecté).
    """
    n = L.shape[0]
    keep = np.array(sorted(set(keep_idx)), dtype=int)
    mask = np.ones(n, dtype=bool)
    mask[keep] = False
    elim = np.where(mask)[0]

    L_kk = L[keep[:, None], keep]
    if elim.size == 0:
        return keep, elim, L_kk.toarray()

    L_ki = L[keep[:, None], elim]
    L_ik = L[elim[:, None], keep]
    L_ii = L[elim[:, None], elim]

    diag = np.array(L_ii.diagonal())
    scale = float(np.mean(diag[diag > 0])) if np.any(diag > 0) else 1.0
    eps = leak_reg * scale
    L_ii_reg = (L_ii + eps * identity(elim.size, format="csc")).tocsc()

    lu = splu(L_ii_reg)
    B = L_ik.toarray()     # (I x K)
    X = lu.solve(B)        # (I x K)

    L_eff = L_kk.toarray() - (L_ki @ X)
    return keep, elim, L_eff


# ============================================================
# 4) Aide géométrique au choix de R
# ============================================================

def geometric_counts(ell, R, lp=1.0, margin_factor=3.0):
    """
    Comptes géométriques rapides (sans stochastique, sans Kron):
      n_int (niveau ℓ) et n_next_int (niveau ℓ+1) pour l’intérieur.
    """
    V, E_geo, pos, radii, coord_of, id_of, s_ell = generate_lattice4_base(ell, R, lp=lp)
    margin = margin_factor * s_ell
    V_int = interior_nodes_by_radius(radii, R, margin)

    blocks, coarse_id_of, IJ_of_coarse, Pi = blocks_2x2(coord_of)
    rad_c = coarse_radii_from_blocks(blocks, coarse_id_of, pos)

    s_next = (2**(ell+1)) * lp
    margin_next = margin_factor * s_next
    Vc_int = interior_nodes_by_radius(rad_c, R, margin_next)

    return dict(
        n_int=len(V_int),
        n_next_int=len(Vc_int),
        s_ell=float(s_ell),
        s_next=float(s_next),
        margin=float(margin),
        margin_next=float(margin_next),
    )

def recommend_R_list(ell_targets=(1,2), R_candidates=range(20, 121, 10), lp=1.0,
                     margin_factor=3.0, n_min_next_geom=200):
    """
    Retourne, pour chaque ℓ cible, les premiers R candidats où n_next_int ≥ n_min_next_geom.
    """
    rec = {}
    for ell in ell_targets:
        ok = []
        for R in R_candidates:
            cnt = geometric_counts(ell, R, lp=lp, margin_factor=margin_factor)
            if cnt["n_next_int"] >= n_min_next_geom:
                ok.append((int(R), int(cnt["n_int"]), int(cnt["n_next_int"])))
        rec[ell] = ok[:10]
    return rec


# ============================================================
# 5) Runner complet v3 : C.5 sur LCC + Kron + diagnostic nonlocal
# ============================================================

def run_C5_lattice4_kron_v3(
    ell_list=(0,1,2),
    R_list=(30, 40, 50, 60),
    delta_tau_list=(1.0, 0.4, 0.2, 0.12, 0.08, 0.06, 0.04, 0.02),
    lp=1.0,
    margin_factor=3.0,
    seed=42,
    # qualité
    n_min=200,
    n_min_next=100,
    chi_min=0.95,
    # calibration percolation
    tau_c=0.08,
    p_c=0.5,
    kappa_keep=1.3,
    # Kron
    leak_reg=1e-9,
    eps_w=1e-12,
):
    rows = []

    for delta_tau in delta_tau_list:
        for R in R_list:
            for ell in ell_list:
                key_int = int(1e6 * float(delta_tau)) + 1000*ell + 10*int(R)
                rng = np.random.default_rng(seed + key_int)

                # ---- fine lattice ----
                V, E_geo, pos, radii, coord_of, id_of, s_ell = generate_lattice4_base(ell, R, lp=lp)

                # ---- stochastique ----
                E, C, st = apply_stochastic_edges(E_geo, rng, delta_tau, tau_c=tau_c, p_c=p_c, kappa_keep=kappa_keep)

                # ---- intérieur fine ----
                margin = margin_factor * s_ell
                V_int = interior_nodes_by_radius(radii, R, margin)
                V_int, E_int = induced_subgraph(V, E, V_int)

                met_full = metrics_full(V_int, E_int)
                met_lcc  = metrics_on_lcc(V_int, E_int)
                inc = incidence_from_conductances(V_int, E_int, C)

                good = int((met_full["n"] >= n_min) and (met_full["lcc_ratio"] >= chi_min))

                # ---- coarse blocks + anchors ----
                blocks, coarse_id_of, IJ_of_coarse, Pi = blocks_2x2(coord_of)
                keep_fine, coarse_of_anchor_fine = choose_anchor_per_block(blocks, coarse_id_of)

                # ---- Kron ----
                L = laplacian_from_conductances(len(V), E, C)
                keep_sorted, elim_sorted, L_eff = kron_reduction(L, keep_fine, leak_reg=leak_reg)

                # map keep_sorted -> coarse ids -> index in L_eff
                coarse_ids_kept = [coarse_of_anchor_fine[fid] for fid in keep_sorted]
                idx_of_coarse = {cid: j for j, cid in enumerate(coarse_ids_kept)}

                # total effective off-diagonal weight (diagnostic)
                K = L_eff.shape[0]
                total_off = 0.0
                for a in range(K):
                    for b in range(a+1, K):
                        w = -L_eff[a, b]
                        if w > eps_w:
                            total_off += float(w)

                # keep only local coarse edges
                E_loc = coarse_local_edges(IJ_of_coarse)
                Cc = {}
                local_off = 0.0
                for (a, b) in E_loc:
                    ia = idx_of_coarse.get(a, None)
                    ib = idx_of_coarse.get(b, None)
                    if ia is None or ib is None:
                        continue
                    w = -L_eff[ia, ib]
                    if w > eps_w:
                        key = (a, b) if a < b else (b, a)
                        Cc[key] = float(w)
                        local_off += float(w)

                nonlocal_frac = float((total_off - local_off) / total_off) if total_off > 0 else float("nan")

                # ---- intérieur coarse ----
                rad_c = coarse_radii_from_blocks(blocks, coarse_id_of, pos)
                s_next = (2**(ell+1)) * lp
                margin_next = margin_factor * s_next

                Vc_all = list(range(len(IJ_of_coarse)))
                Ec_all = list(Cc.keys())

                Vc_int = interior_nodes_by_radius(rad_c, R, margin_next)
                Vc_int, Ec_int = induced_subgraph(Vc_all, Ec_all, Vc_int)

                metc_full = metrics_full(Vc_int, Ec_int)
                metc_lcc  = metrics_on_lcc(Vc_int, Ec_int)
                incc = incidence_from_conductances(Vc_int, Ec_int, Cc)

                good_next = int((metc_full["n"] >= n_min_next) and (metc_full["lcc_ratio"] >= chi_min))

                rows.append({
                    # paramètres
                    "delta_tau": float(delta_tau),
                    "tau_c": float(tau_c),
                    "p_c": float(p_c),
                    "kappa_keep": float(kappa_keep),
                    "p_keep": float(st["p_keep"]),
                    "Aeff": float(st["Aeff"]),
                    "lambda_ov": float(st["lambda_ov"]),
                    "R": float(R),
                    "ell": int(ell),
                    "s_ell": float(s_ell),
                    "margin_factor": float(margin_factor),

                    # fine (full + LCC)
                    "n_int": int(met_full["n"]),
                    "lcc_ratio": float(met_full["lcc_ratio"]),
                    "rho_loop_full": float(met_full["rho_loop"]),
                    "n_lcc": int(met_lcc["n_lcc"]),
                    "rho_loop_lcc": float(met_lcc["rho_loop_lcc"]),
                    "deg_max": float(met_full["deg_max"]),
                    "good": int(good),
                    "m_max": float(inc["m_max"]),

                    # coarse (full + LCC)
                    "n_next_int": int(metc_full["n"]),
                    "lcc_ratio_next": float(metc_full["lcc_ratio"]),
                    "rho_loop_full_next": float(metc_full["rho_loop"]),
                    "n_lcc_next": int(metc_lcc["n_lcc"]),
                    "rho_loop_lcc_next": float(metc_lcc["rho_loop_lcc"]),
                    "deg_max_next": float(metc_full["deg_max"]),
                    "good_next": int(good_next),
                    "m_max_next": float(incc["m_max"]),

                    # Kron nonlocal leakage
                    "nonlocal_frac": float(nonlocal_frac),
                    "total_off_weight": float(total_off),
                    "local_off_weight": float(local_off),
                })

    return pd.DataFrame(rows)

def summarize_C5_v3(df, rho_threshold=0.01):
    """
    Flag polymer_like basé sur les métriques LCC au plus grand R:
      rho_loop_lcc_last ~ 0 AND rho_loop_lcc_next_last ~ 0
    Interpréter uniquement si good/good_next sont non nuls.
    """
    out = []
    for (delta_tau, ell), g in df.groupby(["delta_tau", "ell"]):
        g = g.sort_values("R")
        last = g.iloc[-1]
        polymer_like = bool((last["rho_loop_lcc"] < rho_threshold) and (last["rho_loop_lcc_next"] < rho_threshold))
        out.append({
            "delta_tau": float(delta_tau),
            "ell": int(ell),
            "R_last": float(last["R"]),
            "p_keep_last": float(last["p_keep"]),
            "rho_lcc_last": float(last["rho_loop_lcc"]),
            "rho_lcc_next_last": float(last["rho_loop_lcc_next"]),
            "polymer_like_flag": polymer_like,
            "good_points": int(g["good"].sum()),
            "good_next_points": int(g["good_next"].sum()),
            "lcc_ratio_last": float(last["lcc_ratio"]),
            "lcc_ratio_next_last": float(last["lcc_ratio_next"]),
            "nonlocal_frac_last": float(last["nonlocal_frac"]),
            "m_max_last": float(last["m_max"]),
            "m_max_next_last": float(last["m_max_next"]),
        })
    return pd.DataFrame(out).sort_values(["delta_tau", "ell"]).reset_index(drop=True)


# ============================================================
# 6) Exécution
# ============================================================

margin_factor = 3.0
rec = recommend_R_list(
    ell_targets=(1,2),
    R_candidates=range(20, 121, 10),
    lp=1.0,
    margin_factor=margin_factor,
    n_min_next_geom=200
)
print("Recommandations géométriques (exemples) pour n_next_int ≥ 200 :")
for ell, lst in rec.items():
    print(f"  ℓ={ell} -> (R, n_int, n_next_int) :", lst[:5])

# Choix pratique (ajuster selon rec ci-dessus)
ell_list = (0, 1, 2)
R_list = (30, 40, 50, 60)  # augmenter si tu veux tester ℓ=2→3 avec plus de robustesse
delta_tau_list = (1.0, 0.4, 0.2, 0.12, 0.08, 0.06, 0.04, 0.02)

df = run_C5_lattice4_kron_v3(
    ell_list=ell_list,
    R_list=R_list,
    delta_tau_list=delta_tau_list,
    lp=1.0,
    margin_factor=margin_factor,
    seed=42,
    n_min=200,
    n_min_next=100,
    chi_min=0.95,
    tau_c=0.08,
    p_c=0.5,
    kappa_keep=1.3,
    leak_reg=1e-9,
    eps_w=1e-12
)

summary = summarize_C5_v3(df)

display(df.head(12))
display(summary)

df.to_csv("C5_results_lattice4_kron_v3.csv", index=False)
summary.to_csv("C5_summary_lattice4_kron_v3.csv", index=False)
print("Fichiers générés: C5_results_lattice4_kron_v3.csv, C5_summary_lattice4_kron_v3.csv")

# Concluants seulement
df_good = df[(df["good"] == 1) & (df["good_next"] == 1)]
display(
    df_good.groupby(["delta_tau","ell"])[["p_keep","rho_loop_lcc","rho_loop_lcc_next","nonlocal_frac","m_max","m_max_next"]]
    .mean().reset_index()
)

Recommandations géométriques (exemples) pour n_next_int ≥ 200 :
  ℓ=1 -> (R, n_int, n_next_int) : [(50, 1517, 281), (60, 2289, 451), (70, 3209, 661), (80, 4293, 910), (90, 5525, 1198)]
  ℓ=2 -> (R, n_int, n_next_int) : [(90, 1201, 212), (100, 1517, 281), (110, 1885, 359), (120, 2289, 451)]


,delta_tau,tau_c,p_c,kappa_keep,p_keep,Aeff,lambda_ov,R,ell,s_ell,...,lcc_ratio_next,rho_loop_full_next,n_lcc_next,rho_loop_lcc_next,deg_max_next,good_next,m_max_next,nonlocal_frac,total_off_weight,local_off_weight
0,1.0,0.08,0.5,1.3,0.998596,0.004006,1.212018,30.0,0,1.0,...,1.0,0.895787,451,0.895787,4.0,1,5.740493,0.255875,1851.921249,1378.061810
1,1.0,0.08,0.5,1.3,0.998596,0.004006,1.212018,30.0,1,2.0,...,1.0,0.734375,64,0.734375,4.0,0,4.863612,0.250026,479.261165,359.433432
2,1.0,0.08,0.5,1.3,0.998596,0.004006,1.212018,30.0,2,4.0,...,1.0,0.000000,1,0.000000,0.0,0,0.000000,0.210978,122.425417,96.596298
3,1.0,0.08,0.5,1.3,0.998596,0.004006,1.212018,40.0,0,1.0,...,1.0,0.926374,910,0.926374,4.0,1,6.579766,0.257504,3363.079928,2497.072085
4,1.0,0.08,0.5,1.3,0.998596,0.004006,1.212018,40.0,1,2.0,...,1.0,0.824675,154,0.824675,4.0,1,5.899411,0.245532,848.010054,639.796191
5,1.0,0.08,0.5,1.3,0.998596,0.004006,1.212018,40.0,2,4.0,...,1.0,0.461538,13,0.461538,4.0,0,4.366673,0.239046,205.827498,156.625293
6,1.0,0.08,0.5,1.3,0.998596,0.004006,1.212018,50.0,0,1.0,...,1.0,0.942876,1523,0.942876,4.0,1,6.142906,0.259509,5178.639393,3834.735112
7,1.0,0.08,0.5,1.3,0.998596,0.004006,1.212018,50.0,1,2.0,...,1.0,0.868327,281,0.868327,4.0,1,7.055832,0.253268,1300.232654,970.924859
8,1.0,0.08,0.5,1.3,0.998596,0.004006,1.212018,50.0,2,4.0,...,1.0,0.645161,31,0.645161,4.0,0,4.988611,0.250916,328.232981,245.874213
9,1.0,0.08,0.5,1.3,0.998596,0.004006,1.212018,60.0,0,1.0,...,1.0,0.953357,2294,0.953357,4.0,1,7.098190,0.261750,7485.552211,5526.205416


,delta_tau,ell,R_last,p_keep_last,rho_lcc_last,rho_lcc_next_last,polymer_like_flag,good_points,good_next_points,lcc_ratio_last,lcc_ratio_next_last,nonlocal_frac_last,m_max_last,m_max_next_last
0,0.02,0,60.0,0.026484,0.000000,0.000000,True,0,0,0.000393,0.000872,0.000000,32.0,5.652174
1,0.02,1,60.0,0.026484,0.000000,0.000000,True,0,0,0.001747,0.002217,0.000000,25.0,0.000000
2,0.02,2,60.0,0.026484,0.000000,0.000000,True,0,0,0.006803,0.015625,NaN,22.0,0.000000
3,0.04,0,60.0,0.141586,0.062500,0.000000,False,0,0,0.001570,0.001744,0.048064,49.0,15.984236
4,0.04,1,60.0,0.141586,0.000000,0.000000,True,0,0,0.006553,0.008869,0.042604,52.0,8.181818
5,0.04,2,60.0,0.141586,0.000000,0.000000,True,0,0,0.020408,0.046875,0.055535,34.0,4.562827
6,0.06,0,60.0,0.321263,0.083333,0.000000,False,0,0,0.005889,0.004359,0.121634,62.0,17.765875
7,0.06,1,60.0,0.321263,0.000000,0.100000,False,0,0,0.019222,0.022173,0.113644,58.0,16.285012
8,0.06,2,60.0,0.321263,0.040000,0.000000,False,0,0,0.056689,0.046875,0.082131,50.0,11.610507
9,0.08,0,60.0,0.500000,0.110984,0.188034,False,0,0,0.515556,0.204010,0.182186,64.0,24.335033


Fichiers générés: C5_results_lattice4_kron_v3.csv, C5_summary_lattice4_kron_v3.csv


,delta_tau,ell,p_keep,rho_loop_lcc,rho_loop_lcc_next,nonlocal_frac,m_max,m_max_next
0,0.12,0,0.741582,0.465255,0.835825,0.236169,63.250000,23.689109
1,0.12,1,0.741582,0.449776,0.774851,0.238021,58.000000,22.066730
2,0.20,0,0.915472,0.798701,0.929598,0.254326,45.500000,18.222295
3,0.20,1,0.915472,0.779877,0.862930,0.250373,44.333333,17.157076
4,0.40,0,0.984999,0.937710,0.929598,0.258950,30.250000,11.088461
5,0.40,1,0.984999,0.908174,0.862930,0.255479,27.333333,10.697931
6,1.00,0,0.998596,0.963657,0.929598,0.258660,18.750000,6.390339
7,1.00,1,0.998596,0.936551,0.862930,0.252428,18.000000,6.490261


In [2]:
# ------------------------------------------
# 7) Lecture rapide (concluants seulement)
# ------------------------------------------
df_good = df[(df["good"] == 1) & (df["good_next"] == 1)]
display(
    df_good.groupby(["delta_tau","ell"])[["p_keep","rho_loop_lcc","rho_loop_lcc_next","nonlocal_frac","m_max","m_max_next"]]
    .mean().reset_index()
)

,delta_tau,ell,p_keep,rho_loop_lcc,rho_loop_lcc_next,nonlocal_frac,m_max,m_max_next
0,0.12,0,0.741582,0.465255,0.835825,0.236169,63.250000,23.689109
1,0.12,1,0.741582,0.449776,0.774851,0.238021,58.000000,22.066730
2,0.20,0,0.915472,0.798701,0.929598,0.254326,45.500000,18.222295
3,0.20,1,0.915472,0.779877,0.862930,0.250373,44.333333,17.157076
4,0.40,0,0.984999,0.937710,0.929598,0.258950,30.250000,11.088461
5,0.40,1,0.984999,0.908174,0.862930,0.255479,27.333333,10.697931
6,1.00,0,0.998596,0.963657,0.929598,0.258660,18.750000,6.390339
7,1.00,1,0.998596,0.936551,0.862930,0.252428,18.000000,6.490261
